In [ ]:
import os
import sys
import jax
jax.config.update("jax_enable_x64", True) 

import jax.numpy as jnp
import numpy as np
import qutip as qt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import ExperimentConfig
from src.physics.operators import build_two_qubit_base
from src.devices.qutip_device import HiddenDevice
from src.calibration.fitters import DigitalTwinFitter

import matplotlib.pyplot as plt
from src.calibration.training_loops import make_setup, make_cz_optimizer, train_pulse
from src.experiments.ramsey import (
    reconstruct_pulse_rad,
    evaluate_held_out_probes,
    run_cz_ramsey_experiment,
)
from src.calibration.cz_training import compute_cz_loss
from src.physics.operators import build_true_H0
from src.analysis.metrics import cz_comp_block, cz_process_fidelity

# Load Configurations (Now holding the TRUE hardware parameters)
config = ExperimentConfig.load("configs/two_qubit_CZ_zz.yaml")
dt = config.pulse.dt_ns

# Build static JAX bases (Defaulting to -300.0 to match assemble_H0 scaling)
H_anharm, H_int_base, Hz_control = build_two_qubit_base(
    levels=config.hardware.levels
)


In [ ]:
# Initialize Physical Fridge with Config (True) Hardware Parameters
fridge = HiddenDevice.from_config(config)

# Gather Swap Chevron Measurements (|10> and |11> probes)
dt_step = 8.0
durations_ns = np.array([8.0, 16.0, 24.0, 32.0, 40.0])
amplitudes_ghz = np.linspace(-0.620, -0.570, 7)
amplitudes_rad = 2 * np.pi * amplitudes_ghz

rho_10 = qt.ket2dm(qt.tensor(qt.basis(3, 1), qt.basis(3, 0)))
rho_11 = qt.ket2dm(qt.tensor(qt.basis(3, 1), qt.basis(3, 1)))

target_dataset_10 = np.zeros((len(amplitudes_rad), len(durations_ns), 4))
target_dataset_11 = np.zeros((len(amplitudes_rad), len(durations_ns), 4))

n_shots = config.hardware.n_shots
rng = np.random.default_rng(42)

for i, amp_rad in enumerate(amplitudes_rad):
    for j, dur in enumerate(durations_ns):
        pulse = np.ones(int(dur / dt)) * amp_rad
        target_dataset_10[i, j] = fridge.measure_populations(pulse, rho_10, n_shots=n_shots, rng=rng)
        target_dataset_11[i, j] = fridge.measure_populations(pulse, rho_11, n_shots=n_shots, rng=rng)

# Single Calibration Run from Nominal Config Guesses
fitter = DigitalTwinFitter(dt_step, H_anharm, H_int_base, Hz_control, learning_rate=5e-2, steps=1000)

discovered_params, calib_history = fitter.fit(
    amplitudes_rad, 
    target_dataset_10, 
    target_dataset_11, 
    init_coupling=20.0,    
    init_detuning=600.0,   
    init_alpha=-320.0
)

cal_g = float(discovered_params['coupling_mhz'])
cal_d = float(discovered_params['detuning_mhz'])
cal_a = float(discovered_params['anharmonicity_mhz'])

np.save(config.artifact_path("cal_params.npy"), np.array([cal_g, cal_d, cal_a]))

print("\n--- Calibration Results ---")
print(f"Coupling (g):            {cal_g:.3f} MHz | True: {config.hardware.coupling_mhz:.3f} MHz | Error: {abs(cal_g - config.hardware.coupling_mhz):.3f} MHz")
print(f"Detuning (\\Delta):       {cal_d:.3f} MHz | True: {config.hardware.idle_detuning_mhz:.3f} MHz | Error: {abs(cal_d - config.hardware.idle_detuning_mhz):.3f} MHz")
print(f"Anharmonicity (\\alpha):  {cal_a:.3f} MHz | True: {config.hardware.anharmonicity_mhz:.3f} MHz | Error: {abs(cal_a - config.hardware.anharmonicity_mhz):.3f} MHz")


In [ ]:
cz_steps = 500
optimizer = make_cz_optimizer(0.01, cz_steps)

# Train on Uncalibrated (Naive) Twin
H0_naive, Hz, naive_loss_fn, naive_init, basis, kernel, f_max = make_setup(config, n_harmonics=8, coupling_mhz=20, detuning_mhz=600, anharmonicity_mhz=-300)
naive_opt_params, _ = train_pulse(naive_loss_fn, naive_init, optimizer, cz_steps)

# Train on Calibrated Digital Twin
H0_cal, _, cal_loss_fn, cal_init, _, _, _ = make_setup(
    config, n_harmonics=8, coupling_mhz=cal_g, detuning_mhz=cal_d, anharmonicity_mhz=cal_a
)
cal_opt_params, _ = train_pulse(cal_loss_fn, cal_init, optimizer, cz_steps)

# Cross-validate on 100 held-out commanded random pulses.
# The device applies its hidden 150 MHz response internally; the twin does not.
naive_errors, cal_errors = evaluate_held_out_probes(
    fridge, H0_naive, H0_cal, Hz, dt, basis, kernel, f_max, n_test_pulses=100
)

plt.figure(figsize=(8, 4.5))
bins = np.logspace(-3, 2, 100)
plt.hist(naive_errors, bins=bins, alpha=0.65, color='crimson', label=f'Uncalibrated (Mean: {np.mean(naive_errors):.2f}%)')
plt.hist(cal_errors, bins=bins, alpha=0.75, color='seagreen', label=f'Calibrated (Mean: {np.mean(cal_errors):.4f}%)')
plt.xscale('log')
plt.xlabel('Mean Population Prediction Error (%)')
plt.ylabel('Number of Held-Out Pulses')
plt.title('Held-Out Cross-Validation (100 Random Waveforms)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:


# Reconstruct physical waveforms (rad/ns)
naive_pulse_rad = reconstruct_pulse_rad(naive_opt_params, basis, kernel, f_max)
cal_pulse_rad = reconstruct_pulse_rad(cal_opt_params, basis, kernel, f_max)

# Save BOTH pulses so Notebook 07 can load them directly
np.save(config.artifact_path("cal_pulse_rad.npy"), cal_pulse_rad)
np.save(config.artifact_path("naive_pulse_rad.npy"), naive_pulse_rad)
print(f"Saved optimized CZ pulses to {config.artifact_path('cal_pulse_rad.npy')}")

# Evaluate in the Calibrated Digital Twin's own frame (H0_cal, no hardware RC filter)
_, naive_on_cal_twin = compute_cz_loss(
    naive_opt_params, dt, H0_cal, Hz, basis, kernel, f_max, config.optimizer.leakage_weight
)
_, cal_on_cal_twin = compute_cz_loss(
    cal_opt_params, dt, H0_cal, Hz, basis, kernel, f_max, config.optimizer.leakage_weight
)

print("\n=== 1. Digital Twin Prediction (Evaluated on H0_cal) ===")
print(f"Uncalibrated Pulse -> Twin Fidelity: {(1.0 - float(naive_on_cal_twin['infidelity']))*100:.2f}% | Leakage: {float(naive_on_cal_twin['leakage'])*100:.2f}%")
print(f"Calibrated Pulse   -> Twin Fidelity: {(1.0 - float(cal_on_cal_twin['infidelity']))*100:.4f}% | Leakage: {float(cal_on_cal_twin['leakage'])*100:.4f}%")

# Evaluate on the TRUE Hardware Oracle (H0_true + HiddenDevice RC low-pass filter)
H0_true, Hz_true = build_true_H0(config)
naive_phys = fridge.filter_flux_waveform(naive_pulse_rad)
cal_phys = fridge.filter_flux_waveform(cal_pulse_rad)

oracle_naive = cz_process_fidelity(cz_comp_block(naive_phys, dt, H0_true, Hz_true))
oracle_cal = cz_process_fidelity(cz_comp_block(cal_phys, dt, H0_true, Hz_true))

print("\n=== 2. True Hardware Unitary Performance (H0_true + Cryo Bandwidth Filter) ===")
print(f"Uncalibrated Pulse -> True F_avg: {oracle_naive['F_avg']*100:.2f}% (F_pro: {oracle_naive['F_pro']*100:.2f}%) | Leakage: {oracle_naive['leakage']*100:.2f}%")
print(f"Calibrated Pulse   -> True F_avg: {oracle_cal['F_avg']*100:.2f}% (F_pro: {oracle_cal['F_pro']*100:.2f}%) | Leakage: {oracle_cal['leakage']*100:.2f}%")

# Open-System QuTiP Fridge Verification (With T1/T2 Decoherence + Filter + ZZ)
rho_final_naive = fridge.play_pulse(naive_pulse_rad, rho_11)
rho_final_cal = fridge.play_pulse(cal_pulse_rad, rho_11)

print("\n=== 3. QuTiP Fridge |11> State Verification (With T1/T2 Decoherence) ===")
print(f"Uncalibrated Pulse -> Final P(|11>): {rho_final_naive.diag()[4].real:.4f} | Leakage P(|20>): {rho_final_naive.diag()[6].real:.4f}")
print(f"Calibrated Pulse   -> Final P(|11>): {rho_final_cal.diag()[4].real:.4f} | Leakage P(|20>): {rho_final_cal.diag()[6].real:.4f}")


In [ ]:
phi_sweep, naive_q0, naive_q1 = run_cz_ramsey_experiment(fridge, naive_pulse_rad, naive_opt_params['phases'])
_, cal_q0, cal_q1 = run_cz_ramsey_experiment(fridge, cal_pulse_rad, cal_opt_params['phases'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

ax1.plot(phi_sweep / np.pi, naive_q0, 'o-', label='Control Q1 = $|0\\rangle$', color='royalblue')
ax1.plot(phi_sweep / np.pi, naive_q1, 's-', label='Control Q1 = $|1\\rangle$', color='crimson')
ax1.set_title('Uncalibrated CZ Gate on Hardware')
ax1.set_xlabel('Analysis Phase $\\phi / \\pi$')
ax1.set_ylabel('Qubit 2 Readout Probability $P(|+\\rangle_\\phi)$')
ax1.legend()
ax1.grid(True, alpha=0.4)

ax2.plot(phi_sweep / np.pi, cal_q0, 'o-', label='Control Q1 = $|0\\rangle$', color='royalblue')
ax2.plot(phi_sweep / np.pi, cal_q1, 's-', label='Control Q1 = $|1\\rangle$', color='seagreen')
ax2.set_title('Calibrated CZ Gate on Hardware ($\\pi$ Phase Flip)')
ax2.set_xlabel('Analysis Phase $\\phi / \\pi$')
ax2.legend()
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()
